In [ ]:
%pip install pandas numpy scikit-learn xgboost joblib matplotlib seaborn

In [ ]:
# 1. Install necessary libraries
!pip install -q pandas numpy scikit-learn xgboost joblib

import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, average_precision_score
from google.colab import files # Helper to download files

# --- CONFIGURATION ---
DATA_PATH = '/content/creditcard.csv'
MODEL_FILENAME = 'fraud_model.pkl'
SCALER_FILENAME = 'scaler.pkl'

print("1. Loading Data...")
# Load the dataset
try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    print("ERROR: creditcard.csv not found!")
    raise

# Separate features and target
X = df.drop('Class', axis=1)
y = df['Class']

# Split data (80% Train, 20% Test)
# 2. Splitting and Scaling Data...
print("2. Splitting and Scaling Data...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize the Scaler
scaler = StandardScaler()

# Fit on BOTH columns at once in the Training set
X_train[['Amount', 'Time']] = scaler.fit_transform(X_train[['Amount', 'Time']])

# Transform BOTH columns at once in the Test set (using the learned math)
X_test[['Amount', 'Time']] = scaler.transform(X_test[['Amount', 'Time']])

# Calculate class weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# --- TRAINING ---
print(f"3. Training XGBoost (Weight: {scale_pos_weight:.2f})...")
model = xgb.XGBClassifier(
    objective='binary:logistic',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    scale_pos_weight=scale_pos_weight,
    tree_method='hist',
    device='cuda',      # Uses hella the GPU
    random_state=42
)

model.fit(X_train, y_train)
print("Training Complete!")

# --- EVALUATION ---
print("\n4. Evaluation Results:")
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"AUPRC Score: {average_precision_score(y_test, y_prob):.4f}")

# --- SAVING & DOWNLOADING ---
print("\n5. Saving and Downloading Artifacts...")

# Use the filenames defined at the top (as strings)
joblib.dump(model, MODEL_FILENAME)
joblib.dump(scaler, SCALER_FILENAME)

files.download(MODEL_FILENAME)
files.download(SCALER_FILENAME)


1. Loading Data...
2. Splitting and Scaling Data...
3. Training XGBoost (Weight: 577.29)...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [15:44:37] WARNING: /workspace/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [15:44:37] WARNING: /workspace/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


Training Complete!

4. Evaluation Results:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.33      0.87      0.48        98

    accuracy                           1.00     56962
   macro avg       0.67      0.93      0.74     56962
weighted avg       1.00      1.00      1.00     56962

AUPRC Score: 0.8301

5. Saving and Downloading Artifacts...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>